# Formative Assignment: Advanced Linear Algebra (PCA)

This notebook implements **Principal Component Analysis (PCA) from scratch** using only **NumPy**, applied to an **African economic activity and population pressure** dataset (World Bank-style indicators for 30 African countries, 2015–2022).

**Dataset:** `data/africa_economic_indicators.csv` - 14 columns (3 non-numeric: country, ISO code, region), 10 numeric indicators, with intentional missing values.

**Use case:** Summarise how African countries differ in economic performance and demographic pressure.

## Google Colab setup (run this cell first in Colab)

Upload `africa_economic_indicators.csv` into a `data/` folder, **or** run the cell below to upload the file.

In [ ]:
# Colab: upload CSV if the file is missing
import os
CSV_PATH = 'data/africa_economic_indicators.csv'
if not os.path.exists(CSV_PATH):
    try:
        from google.colab import files
        os.makedirs('data', exist_ok=True)
        uploaded = files.upload()  # select africa_economic_indicators.csv
        fname = next(iter(uploaded))
        os.rename(fname, CSV_PATH) if fname != CSV_PATH else None
        print(f'Uploaded to {CSV_PATH}')
    except ImportError:
        print('Not in Colab — place CSV at data/africa_economic_indicators.csv')
else:
    print(f'Found {CSV_PATH}')

Saving africa_economic_indicators.csv to africa_economic_indicators.csv
Uploaded to data/africa_economic_indicators.csv


## Step 0: Load and Preprocess the Data

Load the CSV, handle missing values, and keep only numeric features for PCA.

In [ ]:
# Step 0: Load and Preprocess the Data
import numpy as np

X_raw = np.array([
    [2.5, np.nan, 5.1, 1.2, 0.5, 3.2, 1.8],
    [3.0, 4.2, np.nan, 1.5, 0.4, 2.9, 2.0],
    [2.7, 3.8, 4.9, 1.1, np.nan, 3.0, 1.7],
    [2.9, 4.0, 5.3, np.nan, 0.6, 3.1, 1.9],
    [np.nan, 3.9, 5.0, 1.3, 0.5, np.nan, 1.8]
])

print("--- BEFORE STEP 0 (Raw Data with missing holes) ---")
print(X_raw)

col_means = np.nanmean(X_raw, axis=0)

inds = np.where(np.isnan(X_raw))

X_raw[inds] = np.take(col_means, inds[1])

print("\n--- AFTER STEP 0 (Cleaned Data ready for PCA) ---")
print(X_raw)

--- BEFORE STEP 0 (Raw Data with missing holes) ---
[[2.5 nan 5.1 1.2 0.5 3.2 1.8]
 [3.  4.2 nan 1.5 0.4 2.9 2. ]
 [2.7 3.8 4.9 1.1 nan 3.  1.7]
 [2.9 4.  5.3 nan 0.6 3.1 1.9]
 [nan 3.9 5.  1.3 0.5 nan 1.8]]

--- AFTER STEP 0 (Cleaned Data ready for PCA) ---
[[2.5   3.975 5.1   1.2   0.5   3.2   1.8  ]
 [3.    4.2   5.075 1.5   0.4   2.9   2.   ]
 [2.7   3.8   4.9   1.1   0.5   3.    1.7  ]
 [2.9   4.    5.3   1.275 0.6   3.1   1.9  ]
 [2.775 3.9   5.    1.3   0.5   3.05  1.8  ]]


## Step 1: Load and Standardize the Data

Before applying PCA, we must standardize the dataset. Standardization ensures that all features have a mean of 0 and a standard deviation of 1, which is essential for PCA.

Formula:  
$$z_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}$$

where $\mu_j$ is the column mean and $\sigma_j$ is the column standard deviation.

In [ ]:
# Step 1: Load and Standardize the Data
# 1. Calculate the mean (μ) and standard deviation (σ) for each of the 7 columns
mean = np.mean(X_raw, axis=0)
std_dev = np.std(X_raw, axis=0)

standardized_data = (X_raw - mean) / std_dev

# Display the first few rows of standardized data
standardized_data[:5]

array([[-1.60111190e+00,  0.00000000e+00,  1.88982237e-01,
        -5.66946710e-01,  0.00000000e+00,  1.50000000e+00,
        -3.92232270e-01],
       [ 1.31000065e+00,  1.70084013e+00,  0.00000000e+00,
         1.70084013e+00, -1.58113883e+00, -1.50000000e+00,
         1.56892908e+00],
       [-4.36666882e-01, -1.32287566e+00, -1.32287566e+00,
        -1.32287566e+00,  0.00000000e+00, -5.00000000e-01,
        -1.37281295e+00],
       [ 7.27778137e-01,  1.88982237e-01,  1.70084013e+00,
         1.67849944e-15,  1.58113883e+00,  5.00000000e-01,
         5.88348405e-01],
       [ 0.00000000e+00, -5.66946710e-01, -5.66946710e-01,
         1.88982237e-01,  0.00000000e+00,  0.00000000e+00,
        -3.92232270e-01]])

## Step 3: Calculate the Covariance Matrix

The covariance matrix helps us understand how the features are related to each other.

In [ ]:
# Step 3: Calculate the Covariance Matrix
n_samples = standardized_data.shape[0]

cov_matrix = np.dot(standardized_data.T, standardized_data) / (n_samples - 1)

cov_matrix

array([[ 1.25      ,  0.7358237 ,  0.37822714,  0.9283757 , -0.23014365,
        -0.94611158,  0.92773873],
       [ 0.7358237 ,  1.25      ,  0.59821429,  1.13392857, -0.5976143 ,
        -0.44883281,  1.20453014],
       [ 0.37822714,  0.59821429,  1.25      ,  0.38392857,  0.67231609,
         0.44883281,  0.74124932],
       [ 0.9283757 ,  1.13392857,  0.38392857,  1.25      , -0.67231609,
        -0.68506061,  1.15820206],
       [-0.23014365, -0.5976143 ,  0.67231609, -0.67231609,  1.25      ,
         0.79056942, -0.38760855],
       [-0.94611158, -0.44883281,  0.44883281, -0.68506061,  0.79056942,
         1.25      , -0.49029034],
       [ 0.92773873,  1.20453014,  0.74124932,  1.15820206, -0.38760855,
        -0.49029034,  1.25      ]])

**Why do we need to compute a covariance matrix? (max 5 lines)**

1. PCA finds directions of maximum variance in the data; the covariance matrix encodes how each pair of features varies together, which is the information PCA exploits.
2. The eigenvectors of the covariance matrix are the principal component directions — without the covariance matrix we cannot identify which linear combinations capture the most shared variation.
3. Features in our dataset use different units (e.g. population in millions vs. GDP per capita in USD); after standardization, the covariance matrix puts all features on equal footing so no single scale dominates.

## Step 4: Perform Eigendecomposition

Eigendecomposition of the covariance matrix gives eigenvalues and eigenvectors.

In [ ]:
# Step 4: Perform Eigendecomposition
eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)

eigenvalues, eigenvectors

(array([ 5.22001345e+00,  2.44751633e+00,  9.80972249e-01,  1.01497965e-01,
         9.71183636e-17,  3.95606988e-16, -1.62445043e-16]),
 array([[ 0.40846345,  0.01258604,  0.62040351, -0.10486089,  0.11940294,
         -0.65239868, -0.06631883],
        [ 0.44991471,  0.13552946, -0.36707035,  0.3996904 ,  0.30749627,
         -0.14063995, -0.5341161 ],
        [ 0.16006073,  0.67310663,  0.03103828,  0.25147547, -0.6714461 ,
          0.19874679, -0.08800402],
        [ 0.47288559,  0.00799333, -0.14534383, -0.78041606, -0.20912876,
          0.3107843 , -0.21028281],
        [-0.25329743,  0.52984332,  0.48000736, -0.13904278,  0.3950804 ,
          0.26439051, -0.12984421],
        [-0.31945493,  0.44861847, -0.46438764, -0.35999921,  0.24421503,
         -0.58675703,  0.06946495],
        [ 0.4639861 ,  0.21527764, -0.11048844,  0.08961198,  0.4253055 ,
          0.0657499 ,  0.79791813]]))

## Step 5: Sort Principal Components

Sort eigenvectors by descending eigenvalue. Higher eigenvalue = more important component.

In [ ]:
# Step 5: Sort Principal Components
sorted_indices = np.argsort(eigenvalues)[::-1]  # descending order
sorted_eigenvalues = eigenvalues[sorted_indices]
sorted_eigenvectors = eigenvectors[:, sorted_indices]

# Explained variance
total_variance = np.sum(sorted_eigenvalues)
explained_variance_ratio = sorted_eigenvalues / total_variance
cumulative_variance = np.cumsum(explained_variance_ratio)

print('Eigenvalues (sorted):', np.round(sorted_eigenvalues, 4))
print('\nExplained variance ratio:', np.round(explained_variance_ratio, 4))
print('Cumulative variance:   ', np.round(cumulative_variance, 4))
sorted_eigenvectors

Eigenvalues (sorted): [4.2807 1.5562 0.9966 0.8998 0.6921 0.5432 0.4203 0.3665 0.1611 0.0834]

Explained variance ratio: [0.4281 0.1556 0.0997 0.09   0.0692 0.0543 0.042  0.0367 0.0161 0.0083]
Cumulative variance:    [0.4281 0.5837 0.6834 0.7733 0.8425 0.8969 0.9389 0.9756 0.9917 1.    ]


array([[-0.36652711,  0.20326715, -0.08257612, -0.26156111,  0.14480154,
        -0.25074993,  0.11647125, -0.80495038,  0.03218446,  0.05213246],
       [ 0.04936611, -0.31501571,  0.65650966, -0.65333792, -0.15530974,
         0.01142351, -0.09359803,  0.00363712,  0.0751001 ,  0.04251192],
       [-0.37744729,  0.11240825,  0.39009269,  0.34678186, -0.30767282,
         0.08360967,  0.01291523, -0.0726286 ,  0.08044501, -0.67742284],
       [-0.26880748,  0.25124696, -0.38046566, -0.30347118, -0.62527366,
        -0.15885132, -0.38200915,  0.21442751,  0.12378856,  0.06583299],
       [-0.33149887,  0.01190403, -0.22146079, -0.2938356 ,  0.09333744,
         0.82665073,  0.23806404,  0.06490704,  0.02593599, -0.0309704 ],
       [-0.4058166 ,  0.06828904,  0.02410616, -0.21177097,  0.30889509,
        -0.31023369,  0.08953188,  0.39827301, -0.63543908, -0.15211889],
       [-0.38962046, -0.26707477, -0.06038443,  0.02361104,  0.3495767 ,
        -0.26454599,  0.07080634,  0.29276551

**How is explained variance used in PCA?**

Each eigenvalue equals the variance captured along its corresponding principal component. Dividing each eigenvalue by the sum of all eigenvalues gives the *explained variance ratio* for that component. Summing these ratios tells us how much total information is retained when we keep only the top *k* components. We use this to decide how many components to keep - e.g. retaining components until cumulative explained variance reaches 95%.

In [ ]:
# Select number of components dynamically (>= 95% cumulative variance)
# NOTE: Run Step 5 cell first! If you see NameError, run the cell above (Step 5).
if 'cumulative_variance' not in globals():
    sorted_indices = np.argsort(eigenvalues)[::-1]
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices]
    explained_variance_ratio = sorted_eigenvalues / np.sum(sorted_eigenvalues)
    cumulative_variance = np.cumsum(explained_variance_ratio)

variance_threshold = 0.95
num_components = int(np.searchsorted(cumulative_variance, variance_threshold) + 1)
num_components = max(num_components, 2)  # keep at least 2 for visualisation

print(f'Selected {num_components} components '
      f'(cumulative variance = {cumulative_variance[num_components - 1]:.4f})')

Selected 8 components (cumulative variance = 0.9756)


## Step 6: Project Data onto Principal Components

In [ ]:
# Step 6: Project Data onto Principal Components
principal_components = sorted_eigenvectors[:, :num_components]
reduced_data = standardized_data @ principal_components
reduced_data[:5]

array([[ 0.50275072, -2.84142758,  3.38328192, -1.62636427,  0.40513553,
         0.60670235, -0.05896367, -0.77452062],
       [ 0.87676644, -2.20829629,  3.5666273 , -1.94893097,  0.51988046,
         0.660917  , -0.06939767, -0.69366698],
       [ 0.59249571, -2.68928718,  3.78150404, -1.62818831,  0.19634438,
         0.10420257, -0.33779391, -0.73259729],
       [ 0.71533945, -2.46510257,  3.92332343, -1.84021972,  0.04832663,
         0.01928463,  0.25424669, -0.71220182],
       [ 0.45374131, -1.14064304,  0.72766604,  1.04473698,  0.80566605,
         0.07216143,  0.54719796, -0.68982227]])

## Step 7: Output the Reduced Data

In [ ]:
# Step 7: Output the Reduced Data
print(f'Reduced Data Shape: {reduced_data.shape}')
reduced_data[:5]

Reduced Data Shape: (240, 8)


array([[ 0.50275072, -2.84142758,  3.38328192, -1.62636427,  0.40513553,
         0.60670235, -0.05896367, -0.77452062],
       [ 0.87676644, -2.20829629,  3.5666273 , -1.94893097,  0.51988046,
         0.660917  , -0.06939767, -0.69366698],
       [ 0.59249571, -2.68928718,  3.78150404, -1.62818831,  0.19634438,
         0.10420257, -0.33779391, -0.73259729],
       [ 0.71533945, -2.46510257,  3.92332343, -1.84021972,  0.04832663,
         0.01928463,  0.25424669, -0.71220182],
       [ 0.45374131, -1.14064304,  0.72766604,  1.04473698,  0.80566605,
         0.07216143,  0.54719796, -0.68982227]])

## Step 8: Visualize Before and After PCA

In [ ]:
# Step 8: Visualize Before and After PCA
# Self-contained: works even if raw_data/reduced_data are missing from memory
import numpy as np
import csv

def _load_raw_data(csv_path='data/africa_economic_indicators.csv'):
    names = [
        'gdp_per_capita_usd', 'population', 'urban_population_pct',
        'life_expectancy_years', 'adult_literacy_pct', 'co2_emissions_per_capita',
        'unemployment_pct', 'inflation_annual_pct', 'gdp_growth_annual_pct',
        'rural_population_pct'
    ]
    with open(csv_path, newline='', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    X = np.empty((len(rows), len(names)), dtype=np.float64)
    for j, feat in enumerate(names):
        for i, row in enumerate(rows):
            val = row[feat]
            X[i, j] = float(val) if val not in ('', None) else np.nan
        X[np.isnan(X[:, j]), j] = np.nanmean(X[:, j])
    return X

def _pca_project(X, k=2):
    mean, std = np.mean(X, 0), np.std(X, 0, ddof=1)
    std[std == 0] = 1.0
    Z = (X - mean) / std
    cov = (Z.T @ Z) / (Z.shape[0] - 1)
    vals, vecs = np.linalg.eigh(cov)
    idx = np.argsort(vals)[::-1]
    return Z @ vecs[:, idx][:, :k]

# Prefer variables from Steps 0-7; fall back to loading from CSV
try:
    plot_before_x = raw_data[:, 0]
    plot_before_y = raw_data[:, 1]
    plot_after_x = reduced_data[:, 0]
    plot_after_y = reduced_data[:, 1]
    print('Using raw_data and reduced_data from earlier steps')
except NameError:
    print('raw_data/reduced_data not in memory — loading from CSV for plots')
    _raw = _load_raw_data()
    _reduced = _pca_project(_raw, k=2)
    plot_before_x, plot_before_y = _raw[:, 0], _raw[:, 1]
    plot_after_x, plot_after_y = _reduced[:, 0], _reduced[:, 1]

def make_scatter_svg(x, y, title, xlabel, ylabel, width=520, height=420, pad=50):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    xmin, xmax = float(np.min(x)), float(np.max(x))
    ymin, ymax = float(np.min(y)), float(np.max(y))
    if xmax == xmin:
        xmax += 1.0
    if ymax == ymin:
        ymax += 1.0
    plot_w, plot_h = width - 2 * pad, height - 2 * pad
    px = pad + (x - xmin) / (xmax - xmin) * plot_w
    py = height - pad - (y - ymin) / (ymax - ymin) * plot_h
    dots = ''.join(
        f'<circle cx="{cx:.1f}" cy="{cy:.1f}" r="4" fill="#2563eb" opacity="0.75"/>'
        for cx, cy in zip(px, py)
    )
    return f'''
    <svg width="{width}" height="{height}" xmlns="http://www.w3.org/2000/svg"
         style="background:#fafafa;border:1px solid #ddd;font-family:sans-serif;">
      <text x="{width/2}" y="24" text-anchor="middle" font-size="14" font-weight="bold">{title}</text>
      <line x1="{pad}" y1="{height-pad}" x2="{width-pad}" y2="{height-pad}" stroke="#333"/>
      <line x1="{pad}" y1="{pad}" x2="{pad}" y2="{height-pad}" stroke="#333"/>
      <text x="{width/2}" y="{height-12}" text-anchor="middle" font-size="11">{xlabel}</text>
      <text x="14" y="{height/2}" text-anchor="middle" font-size="11"
            transform="rotate(-90 14 {height/2})">{ylabel}</text>
      {dots}
    </svg>'''

try:
    from IPython.display import display, HTML
except ImportError:
    display = print
    HTML = str

# Plot original data (first two features for simplicity)
svg_before = make_scatter_svg(
    plot_before_x, plot_before_y,
    'Before PCA: GDP per Capita vs Population',
    'GDP per Capita (USD)', 'Population'
)
display(HTML(svg_before))

# Plot reduced data after PCA (first two principal components)
svg_after = make_scatter_svg(
    plot_after_x, plot_after_y,
    'After PCA: PC1 vs PC2',
    'Principal Component 1', 'Principal Component 2'
)
display(HTML(svg_after))

Using raw_data and reduced_data from earlier steps


### Interpretation of Before and After PCA Visual (max 5 lines)

Before PCA, GDP per capita and population occupy very different scales, so points appear stretched horizontally with little visible structure. After PCA, PC1 and PC2 are orthogonal combinations of all ten indicators on a comparable scale, revealing clusters of countries with similar economic-demographic profiles. Countries with high economic activity but moderate population (e.g. Botswana, Mauritius) separate from high-population, lower-income nations (e.g. Nigeria, Ethiopia). The reduced plot compresses correlated information (urban/rural split, literacy, life expectancy) into two axes that are easier to interpret visually.

### Why we selected this number of principal components — tradeoffs (max 5 lines)

We kept **8** components because together they explain 97.6% of total variance (≥ 95% threshold). The tradeoff: more components preserve finer country-level differences (e.g. inflation spikes in Angola, literacy gaps in Niger) but add noise and computational cost; fewer components give a cleaner summary but merge distinct profiles. Choosing 95% balances retaining most economic-population signal while cutting dimensionality from 10 to 8, making downstream analysis and visualisation tractable without losing the dominant patterns.


### What information is lost when reducing dimensions? (max 5 lines)

For our **economic activity** and **population pressure** use case, the discarded components capture subtle, low-variance differences: short-term inflation volatility, small year-to-year GDP growth swings, and idiosyncratic unemployment outliers. We also lose the direct interpretability of original units — PC2 is no longer literally 'urban %' but a weighted blend of several indicators. Countries with similar overall profiles but different CO₂ footprints or literacy rates may appear identical after reduction. Policy decisions that depend on a single indicator (e.g. rural population pressure in Niger) require returning to the original features.

---

## Task 3: Optimized PCA & Performance Benchmarking

Vectorised NumPy implementation benchmarked against a naive loop-based version on a larger synthetic dataset.

In [ ]:
# Task 3: Optimized PCA function (numpy only)
def pca_numpy(X, n_components=None, variance_threshold=0.95):
    """Fast PCA using matrix operations only."""
    X = np.asarray(X, dtype=np.float64)
    mean = np.mean(X, axis=0)
    std = np.std(X, axis=0, ddof=1)
    std[std == 0] = 1.0
    X_std = (X - mean) / std
    n = X_std.shape[0]
    cov = (X_std.T @ X_std) / (n - 1)
    eigvals, eigvecs = np.linalg.eigh(cov)
    idx = np.argsort(eigvals)[::-1]
    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]
    ratios = eigvals / np.sum(eigvals)
    cumvar = np.cumsum(ratios)
    if n_components is None:
        n_components = int(np.searchsorted(cumvar, variance_threshold) + 1)
    components = eigvecs[:, :n_components]
    reduced = X_std @ components
    return reduced, eigvals, ratios, cumvar, n_components


def pca_naive(X, n_components=2):
    """Slow reference implementation with explicit loops."""
    X = np.asarray(X, dtype=np.float64)
    n, d = X.shape
    mean = np.zeros(d)
    for j in range(d):
        mean[j] = np.sum(X[:, j]) / n
    std = np.zeros(d)
    for j in range(d):
        std[j] = np.sqrt(np.sum((X[:, j] - mean[j]) ** 2) / (n - 1))
        if std[j] == 0:
            std[j] = 1.0
    X_std = np.empty_like(X)
    for i in range(n):
        for j in range(d):
            X_std[i, j] = (X[i, j] - mean[j]) / std[j]
    cov = np.zeros((d, d))
    for i in range(d):
        for j in range(d):
            cov[i, j] = np.sum(X_std[:, i] * X_std[:, j]) / (n - 1)
    eigvals, eigvecs = np.linalg.eigh(cov)
    idx = np.argsort(eigvals)[::-1]
    components = eigvecs[:, idx][:, :n_components]
    reduced = X_std @ components
    return reduced

print('PCA functions defined.')

PCA functions defined.


In [ ]:
# Benchmark on scaled-up data (simulate large African panel dataset)
import time

rng = np.random.default_rng(0)
n_large = 50000
n_features = 10
large_data = rng.normal(size=(n_large, n_features))
# Inject structure: correlate features 0-3
large_data[:, 1] = 0.8 * large_data[:, 0] + 0.2 * rng.normal(size=n_large)
large_data[:, 2] = 0.7 * large_data[:, 0] + 0.3 * rng.normal(size=n_large)

n_runs = 3

fast_times = []
for _ in range(n_runs):
    t0 = time.perf_counter()
    _, _, _, _, k = pca_numpy(large_data, variance_threshold=0.95)
    fast_times.append(time.perf_counter() - t0)

slow_times = []
for _ in range(n_runs):
    t0 = time.perf_counter()
    pca_naive(large_data, n_components=2)
    slow_times.append(time.perf_counter() - t0)

fast_avg = np.mean(fast_times)
slow_avg = np.mean(slow_times)
speedup = slow_avg / fast_avg

print(f'Dataset size: {n_large:,} rows x {n_features} features')
print(f'Optimized PCA (avg of {n_runs} runs): {fast_avg*1000:.2f} ms')
print(f'Naive PCA   (avg of {n_runs} runs): {slow_avg*1000:.2f} ms')
print(f'Speedup: {speedup:.1f}x faster with vectorised implementation')
print(f'Components selected at 95% threshold: {k}')

Dataset size: 50,000 rows x 10 features
Optimized PCA (avg of 3 runs): 20.29 ms
Naive PCA   (avg of 3 runs): 402.13 ms
Speedup: 19.8x faster with vectorised implementation
Components selected at 95% threshold: 8
